# PriceLens Text Baseline Demo

Use this notebook to validate the data cleaning, split, TF-IDF feature extraction, Ridge regression baseline, and metrics before promoting the workflow into production scripts.

## Dataset Contract

- `data/raw/train.csv`: `sample_id`, `catalog_content`, `image_link`, `price`
- `data/raw/test.csv`: `sample_id`, `catalog_content`, `image_link`
- `data/images/`: images referenced by `image_link`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

TRAIN_PATH

In [ ]:
df = pd.read_csv(TRAIN_PATH)
df.head()

In [ ]:
required_columns = {"sample_id", "catalog_content", "image_link", "price"}
missing_columns = required_columns - set(df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

clean = df.copy()
clean["catalog_content"] = clean["catalog_content"].fillna("").astype(str).str.strip()
clean["price"] = pd.to_numeric(clean["price"], errors="coerce")
clean = clean[(clean["catalog_content"] != "") & clean["price"].notna() & (clean["price"] > 0)]
clean = clean.drop_duplicates(subset=["sample_id"])

clean.shape, clean["price"].describe()

In [ ]:
train_df, valid_df = train_test_split(clean, test_size=0.2, random_state=42)

pipeline = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(max_features=80_000, ngram_range=(1, 2), min_df=2)),
        ("model", Ridge(alpha=1.0)),
    ]
)

pipeline.fit(train_df["catalog_content"], np.log1p(train_df["price"]))
valid_pred = np.expm1(pipeline.predict(valid_df["catalog_content"]))
valid_pred = np.clip(valid_pred, a_min=0, a_max=None)

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    values = np.where(denominator == 0, 0, np.abs(y_true - y_pred) / denominator)
    return float(np.mean(values) * 100)

metrics = {
    "SMAPE": smape(valid_df["price"], valid_pred),
    "MAE": mean_absolute_error(valid_df["price"], valid_pred),
    "RMSE": np.sqrt(mean_squared_error(valid_df["price"], valid_pred)),
    "R2": r2_score(valid_df["price"], valid_pred),
}

metrics

In [ ]:
vectorizer = pipeline.named_steps["tfidf"]
model = pipeline.named_steps["model"]
feature_names = np.asarray(vectorizer.get_feature_names_out())
top_positive = feature_names[np.argsort(model.coef_)[-20:]][::-1]
top_negative = feature_names[np.argsort(model.coef_)[:20]]

pd.DataFrame({"positive_price_signal": top_positive, "negative_price_signal": top_negative})

In [ ]:
report = ["# Baseline Metrics", ""]
for name, value in metrics.items():
    report.append(f"- {name}: {value:.4f}")

(REPORTS_DIR / "baseline_metrics.md").write_text("\n".join(report), encoding="utf-8")